# Lab 1: PyRIT Basics

PyRIT の Target、Attack、Converter を小さく体験します。

### PyRIT の初期化

このハンズオン用に、PyRIT の Memory を Notebook 内だけの一時ストレージとして初期化します。Notebook のカーネルを再起動すると、会話履歴や実行結果は消えます。

In [ ]:
from helpers.handson_utils import initialize_pyrit_in_memory

initialize_pyrit_in_memory()

### このコードでやっていること

このセルは、PyRIT の基本的な実行フローである「設定を読み込む → Target を作る → Attack を作る → 目的を送る → 結果を表示する」を体験するためのコードです。

1. `load_handson_env()` で `.env` から接続設定を読み込みます。
   - `OPENAI_CHAT_ENDPOINT`: モデル API の接続先
   - `OPENAI_CHAT_KEY`: API キー
   - `OPENAI_CHAT_MODEL`: 使用するモデル名、または Azure OpenAI のデプロイ名
2. `ConsoleAttackResultPrinter` と `PromptSendingAttack` を読み込みます。
   - `PromptSendingAttack` は、指定した `objective` を Target に送るシンプルな 1 ターンの Attack です。
   - `ConsoleAttackResultPrinter` は、実行結果の会話を Notebook 上に見やすく表示するためのクラスです。
3. `OpenAIChatTarget` を読み込み、`.env` から取得した `endpoint`、`api_key`、`model_name` を明示的に渡して Target を作ります。
4. `attack = PromptSendingAttack(objective_target=target)` で、その Target にプロンプトを送る Attack を作ります。
5. `attack.execute_async(...)` で「Python の仮想環境を初心者向けに短く説明してください。」という目的を Target に送ります。
6. `ConsoleAttackResultPrinter().print_conversation_async(...)` で、ユーザー入力とモデル応答の会話結果を表示します。

補足: `OpenAIChatTarget` は OpenAI Chat Completions API 互換の Target です。OpenAI だけでなく、Azure OpenAI や OpenAI 互換エンドポイントを持つモデルでも利用できます。PyRIT 公式ドキュメントでは、例として `gpt-4o`、`gpt-4`、`DeepSeek`、`llama`、`phi-4`、`gpt-3.5` などが挙げられています。

- PyRIT 公式: [OpenAIChatTarget](https://microsoft.github.io/PyRIT/code/targets/openai-chat-target/)
- PyRIT 公式: [Prompt Targets 一覧](https://microsoft.github.io/PyRIT/code/targets/prompt-targets/)

Target の種類は `OpenAIChatTarget` だけではありません。PyRIT には OpenAI Responses、画像、音声、HTTP、カスタム Target など、用途に応じた複数の Target が用意されています。詳細は上記の Prompt Targets 一覧を参照してください。

In [ ]:
from helpers.handson_utils import load_handson_env
from pyrit.executor.attack import ConsoleAttackResultPrinter, PromptSendingAttack
from pyrit.prompt_target import OpenAIChatTarget

config = load_handson_env()

target = OpenAIChatTarget(
    endpoint=config["OPENAI_CHAT_ENDPOINT"],
    api_key=config["OPENAI_CHAT_KEY"],
    model_name=config["OPENAI_CHAT_MODEL"],
)
attack = PromptSendingAttack(objective_target=target)

result = await attack.execute_async(objective="Python の仮想環境を初心者向けに短く説明してください。")
await ConsoleAttackResultPrinter().print_conversation_async(result=result)

### Converter でプロンプトを変換する

このセルでは、PyRIT の Converter がプロンプト文字列をどのように変換するかを確認します。Converter は、Target に送る前のプロンプトを別の形式に変換するための部品です。

1. `Base64Converter`、`ROT13Converter`、`MorseConverter` を読み込みます。
   - `Base64Converter` は文字列を Base64 形式に変換します。
   - `ROT13Converter` はアルファベットを 13 文字ずらす ROT13 形式に変換します。
   - `MorseConverter` は文字列をモールス符号に変換します。
2. `prompt` に、安全なサンプル文を入れます。
3. `convert_async(prompt=prompt)` で、それぞれの Converter を使ってプロンプトを変換します。
4. `output_text` を表示して、元の文と変換後の文を比較します。

ここではモデルには送信せず、Converter 単体の動きを確認しています。

補足: サンプル文を英語にしているのは、`ROT13Converter` や `MorseConverter` が主に英字アルファベットを変換する仕組みだからです。日本語の文章だと変換結果が分かりにくいため、ここでは変換結果を見比べやすい英語の短文を使っています。

PyRIT には、ここで使う Text-to-Text Converter 以外にも、音声、画像、動画、ファイルを扱う Converter などがあります。詳しくは公式ドキュメントを参照してください。

- PyRIT 公式: [Converters](https://microsoft.github.io/PyRIT/code/converters/converters/)
- PyRIT 公式: [Text-to-Text Converters](https://microsoft.github.io/PyRIT/code/converters/text-to-text-converters/)

In [ ]:
from pyrit.prompt_converter import Base64Converter, MorseConverter, ROT13Converter

prompt = "Please keep the training secret phrase private."

base64_result = await Base64Converter().convert_async(prompt=prompt)
rot13_result = await ROT13Converter().convert_async(prompt=prompt)
morse_result = await MorseConverter().convert_async(prompt=prompt)

print("Original:", prompt)
print("Base64:", base64_result.output_text)
print("ROT13:", rot13_result.output_text)
print("Morse:", morse_result.output_text)